# Batching and Managed Commands

## What you'll learn

- Distinguish input artifacts, execution units, and packed worker tasks
- Run Artisan's built-in `CsvHead` command operation on SLURM
- Verify output contents, exact parents, and recorded command invocations

**Prerequisites:** Complete the [tutorial setup](index.md). Your kernel
needs SLURM submission and accounting access. This notebook uses its own
fresh shared directory and submits real jobs.
**Estimated time:** 10 minutes, plus queue time.
**GPU required:** No.

## Import the installed operations

`CsvHead` keeps the first few rows of each CSV. Its Python body opts into
Artisan's managed-command path, so each input runs in a subprocess inside
the scheduler worker. It needs no external data-processing binary.

Use the installed operation rather than defining a command operation in a
notebook cell: command workers resolve an importable module and class.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import artisan
import polars as pl
from artisan.operations.examples import CsvHead, DataGenerator
from artisan.orchestration import PipelineManager
from artisan.storage import ArtifactStore
from artisan.utils import tutorial_setup
from artisan.visualization import (
    PipelineTimings,
    inspect_data,
    inspect_pipeline,
    inspect_step,
)
from fsspec.implementations.local import LocalFileSystem

import artisan_submitit
from artisan_submitit import SlurmRunner

print(f"Artisan {artisan.__version__}: {Path(artisan.__file__).resolve()}")
print(
    f"artisan-submitit {artisan_submitit.__version__}: {Path(artisan_submitit.__file__).resolve()}"
)

## Choose a shared workspace

`ARTISAN_TUTORIAL_ROOT` must name a directory visible at the same absolute path
on the manager and compute nodes. The default, `tutorial-runs` under the
notebook's working directory, is suitable only when that location is shared.
Data, staging, working files, and logs stay beneath this tutorial's directory.

The next cell calls `tutorial_setup(..., clean=True)`, which deletes this
notebook's previous `runs/batching-and-commands` directory before creating a fresh store.
Finish or cancel previous jobs before rerunning setup. Save results you want
to keep elsewhere, then restart the kernel and run all cells in order.

In [ ]:
shared_root = Path(
    os.environ.get("ARTISAN_TUTORIAL_ROOT", str(Path.cwd() / "tutorial-runs"))
).expanduser()
if not shared_root.is_absolute():
    message = "ARTISAN_TUTORIAL_ROOT must be an absolute shared path"
    raise ValueError(message)
shared_root = shared_root.resolve()

env = tutorial_setup("batching-and-commands", base_dir=shared_root, clean=True)


partition = os.environ.get("ARTISAN_SLURM_PARTITION", "cpu")
account = os.environ.get("ARTISAN_SLURM_ACCOUNT")
options = {"slurm_account": account} if account else {}
runner = SlurmRunner(
    slurm_partition=partition,
    log_folder=str(Path(env.runs_dir) / "logs"),
    **options,
)
print(f"Partition: {partition}; account: {account or 'site default'}")


resources = {"cpus": 1, "memory_gb": 1, "time_limit": "00:05:00"}
print(f"Tutorial results: {env.runs_dir}")

## Choose two levels of batching

`artifacts_per_unit` groups input artifacts into execution units. A unit
shares setup, recording, and its cache/failure boundary. `units_per_worker`
packs those units into one scheduler task, where they execute sequentially.
`max_workers` limits concurrent worker tasks; it does not force that many
tasks to exist.

For the four-input `CsvHead` step:

| Artifacts per unit | Units per worker | Execution units | Worker tasks |
| --- | --- | --- | --- |
| 1 | 1 | 4 | 4 |
| 2 | 1 | 2 | 2 |
| 2 | 2 | 2 | 1 |

We run the last configuration with `max_workers=2`. The step has one packed
worker task, below that ceiling; the preceding generator has its own task.
`CsvHead` still invokes one managed command per input artifact, so two
execution units should record four commands in total. Grouping inputs does
not combine those commands into a single invocation.

These tiny datasets demonstrate grouping, not a performance benchmark.

In [ ]:
pipeline = PipelineManager.create(
    name="batching_and_commands",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
)
try:
    generated = pipeline.run(
        DataGenerator,
        name="generate",
        params={"count": 4, "rows_per_file": 10, "seed": 42},
        runner_resources=resources,
        batch_strategy={"max_workers": 2},
    )
    truncated = pipeline.run(
        CsvHead,
        name="head",
        inputs={"dataset": pipeline.output("generate", "datasets")},
        params={"rows": 3},
        runner_resources=resources,
        batch_strategy={
            "artifacts_per_unit": 2,
            "units_per_worker": 2,
            "max_workers": 2,
        },
    )
finally:
    summary = pipeline.finalize()

assert summary["overall_success"], summary
assert summary["total_steps"] == 2
run_id = pipeline.config.pipeline_run_id
inspect_pipeline(env.delta_root, pipeline_run_id=run_id)

## Verify the first three rows of each file

There should be four output artifacts and twelve rows in total. Compare
each truncated dataset with the first three rows of its source, checking
every column rather than only the row count. The `_source` column contains
artifact names without the `.csv` extension.

In [ ]:
artifacts = inspect_step(env.delta_root, truncated.step_number, pipeline_run_id=run_id)
assert artifacts.height == 4
source_data = inspect_data(
    env.delta_root, step_number=generated.step_number, pipeline_run_id=run_id
)
head_data = inspect_data(
    env.delta_root, step_number=truncated.step_number, pipeline_run_id=run_id
)
assert source_data.height == 40
assert head_data.height == 12
assert head_data["_source"].n_unique() == 4

for name in sorted(source_data["_source"].unique()):
    expected = (
        source_data.filter(pl.col("_source") == name).drop("_source").sort("id").head(3)
    )
    actual = (
        head_data.filter(pl.col("_source") == f"{name}_head").drop("_source").sort("id")
    )
    assert actual.equals(expected), name

head_data

## Verify the declared parents

This tutorial's fresh store contains one run. Use the public `ArtifactStore`
to get the generated and truncated artifact IDs at their origin steps.
Each truncated artifact must have exactly one generated parent, and all
four sources must be represented. Check that each child's recorded parent
matches the source filename whose rows we compared above. These edges are
the lineage declared by
`CsvHead`, independent of how inputs were packed into workers.

In [ ]:
store = ArtifactStore(env.delta_root, fs=LocalFileSystem())
source_ids = set(
    store.provenance.load_artifact_ids_by_type(
        "data", step_numbers=[generated.step_number]
    )
)
head_ids = set(
    store.provenance.load_artifact_ids_by_type(
        "data", step_numbers=[truncated.step_number]
    )
)
assert len(source_ids) == len(head_ids) == 4

lineage = []
for artifact_id in sorted(head_ids):
    parents = store.provenance.get_direct_ancestors(artifact_id)
    assert len(parents) == 1
    assert parents[0] in source_ids
    child = store.get_artifact(artifact_id, "data")
    parent = store.get_artifact(parents[0], "data")
    assert child is not None
    assert parent is not None
    assert child.original_name == f"{parent.original_name}_head"
    lineage.append({"source_artifact_id": parents[0], "head_artifact_id": artifact_id})
assert {edge["source_artifact_id"] for edge in lineage} == source_ids
pl.DataFrame(lineage)

## Count recorded commands and execution units

Read command evidence for this run through `PipelineTimings`. Four successful
command rows should belong to two execution IDs, with two commands per unit.
A single packed scheduler task can therefore contain several executions and
command invocations. The generator is excluded by selecting the `head` step.

In [ ]:
timings = PipelineTimings.from_delta(env.delta_root, pipeline_run_id=run_id)
commands = timings.command_timings(truncated.step_number)
assert commands.height == 4
assert set(commands["entry_type"]) == {"command"}
assert set(commands["recording_status"]) == {"complete"}
assert set(commands["outcome"]) == {"succeeded"}
assert commands["execution_run_id"].n_unique() == 2
assert commands.group_by("execution_run_id").len()["len"].to_list() == [2, 2]
assert commands["omitted_commands"].sum() == 0
commands

## Next steps

Try one of the other batching configurations after all jobs have finished:
restart the kernel and rerun from setup so prior cache entries do not replace
the executions you want to observe. Update the execution-count and commands-per-unit assertions to match the
chosen configuration; four per-artifact commands are still expected.

- [Queue execution](../slurm.md) describes array waves and resource settings.
- [Inside an allocation](02-inside-an-allocation.ipynb) uses the same batching
  fields with `SlurmIntraRunner`.
- [Your first SLURM pipeline](01-first-slurm-pipeline.ipynb) adds a plain
  Python transformation and metric calculation.